## **AI And Biotechnology/Bioinformatics**

## **AI and Drug Discovery Course: QSAR Modeling**
This notebook demonstrates how to build **Random forest Regression model** to predict pIC50 values.

When you parse molecular structures into thousands of structural variables (like PaDEL-Descriptor, Mordred, or RDKit fingerprints) to predict a continuous biological value (like $pIC_{50}$), Random Forest handles the unique challenges of chemical data exceptionally well.

In [ ]:
# Install LazyPredict
!pip install lazypredict


# **SHAP:**

At its core, **SHAP** **(SHapley Additive exPlanations) **is a translation tool for machine learning.

It takes complex, machine learning models (like Random Forest, XGBoost, or Neural Networks) and breaks them down so humans can understand exactly why a model made a specific prediction.

If your machine learning model is the "engine,"SHAP is the dashboard that tells you exactly how much horsepower every single part is contributing.

In [ ]:
# Install SHAP
!pip install shap

In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.utils import shuffle
from lazypredict.Supervised import LazyRegressor
import shap

## **Loading the Dataset**

In [ ]:
# Upload CSV file from your computer
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv('QSAR_dataset.csv')
df.head()

## **Dataset overview:**
- `molecule_chembl_id`: Unique molecule ID
- `bioactivity_class`: Active/inactive class
- `pIC50`: Continuous potency value (4-9)
- fingerprints






## **Features (X) and Target (y)**

Features are our PubChem fingerprints. Target is pIC50 for regression.
We'll exclude non-feature columns first.

In [ ]:
df = pd.read_csv('QSAR_dataset.csv')
# Exclude non-feature columns
non_feature_cols = ['molecule_chembl_id', 'bioactivity_class', 'pIC50']
X = df.drop(columns=non_feature_cols)
print(X.shape)

In [ ]:
# Target variable
y_reg = df['pIC50']
y_reg

## **Feature Selection – Variance Threshold**

Not all 881 fingerprints are informative. Low-variance features (mostly 0 or 1) add noise. We remove them using `VarianceThreshold`.

If a column contains almost the exact same value for every single row, it has low "variance." Because the values don't vary, that column cannot help a machine learning model tell the difference between an active compound and an inactive one—it just adds unnecessary noise and computation.

For binary columns like your PubChem fingerprints, the variance is calculated using the formula:
$$\text{Variance} = p(1 - p)$$

the threshold is set to 0.8 * (1 - 0.8). This explicitly tells the computer: "Look at each of the 881 fingerprint columns. If a column contains more than **80% zeros** OR more than **80% ones**, drop it completely."

In [ ]:
# Apply variance threshold
selection = VarianceThreshold(threshold=(0.8*(1-0.8)))  # Threshold = 0.16
X_var = selection.fit_transform(X)

# Extract the correct feature names
selected_mask = selection.get_support()
feature_names = X.columns[selected_mask]

# Convert immediately to DataFrame
X_var = pd.DataFrame(X_var, columns=feature_names)

print('After variance threshold:', X_var.shape)

In [ ]:
881 - 151

**Result:** 151 informative features retained, 730 removed. Reduces noise, speeds up training, and improves generalization.

## **Split Data into Training and Test Set**

**(The 80/20 Rule):**

This dictates that exactly 20% of your molecules are randomly selected and locked away as the testing set. The remaining 80% are assigned to the training set.

**The Inputs Supplied to Split**

**X_var:** Your clean features matrix, containing highly informative PubChem fingerprint columns we selected in the previous step.

**y_reg:** Your regression target array, which holds the continuous $pIC_{50}$ biological activity values.

In [ ]:
# Split into 80/20 train/test
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_var, y_reg, test_size=0.2, random_state=123
)
print(f'Training set size: {X_train_reg.shape[0]} molecules')
print(f'Testing set size: {X_test_reg.shape[0]} molecules')

## **Build and Train Random Forest Model**

In [ ]:
import numpy as np

# Set random seed for reproducibility
np.random.seed(123)

# Filter out infinite or excessively large values from y_train_reg
# The problem states pIC50 values are usually 4-9, so anything very large or infinite is an anomaly.
# We will use np.isfinite to check for valid numbers.
finite_mask = np.isfinite(y_train_reg)
X_train_reg_filtered = X_train_reg[finite_mask]
y_train_reg_filtered = y_train_reg[finite_mask]

print(f"Original training set size: {len(y_train_reg)}")
print(f"Filtered training set size: {len(y_train_reg_filtered)}")

# Create Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=200, random_state=123) # Added random_state for reproducibility consistent with np.random.seed

# Train model with filtered data
rf_model.fit(X_train_reg_filtered, y_train_reg_filtered)

## **Evaluate Model's Performance on Test Set**

In [ ]:
r2 = rf_model.score(X_test_reg, y_test_reg)
print(f'Random Forest R² score: {r2:.4f}')

## **R² interpretation:**

$R^2$ (pronounced R-squared), also known as the Coefficient of Determination.

It measures how well your Random Forest model's predicted $pIC_{50}$ values match up with the actual experimental $pIC_{50}$ values.
- **R² < 0.4** → Poor fit.
- **R² < 0.5-0.7** → Moderate/Acceptable.
- **R² > 0.9** → Very good/High fit

Other metrics like **RMSE**(Root Mean Squared Error) or **MAE**(Mean Absolute Error) can also be used.

## **Make Predictions on test set**

In [ ]:
# Predict on test set
y_pred_reg = rf_model.predict(X_test_reg)

In [ ]:
y_pred_reg

**y_test_reg**       ..............# actual pIC50  
**y_pred_reg**       ..............# predicted pIC50 values from model

## **Visualize Predictions vs Actual Values**



In [ ]:
ax = sns.regplot(x=y_test_reg, y=y_pred_reg, scatter_kws={'alpha':0.4})
ax.set_xlabel('Experimental pIC50', fontsize='large', fontweight='bold')
ax.set_ylabel('Predicted pIC50', fontsize='large', fontweight='bold')
plt.show()

## **Y-Randomization – Validate Model**

In [ ]:
n_iterations = 5
random_r2_scores = []

for i in range(n_iterations):
    # Shuffle the *filtered* y_train_reg
    y_train_shuffled = shuffle(y_train_reg_filtered, random_state=i)
    rf_random = RandomForestRegressor(n_estimators=200, random_state=42)
    # Fit using the filtered X_train_reg and the shuffled filtered y_train_reg
    rf_random.fit(X_train_reg_filtered, y_train_shuffled)
    y_pred_random = rf_random.predict(X_test_reg)
    r2_random = r2_score(y_test_reg, y_pred_random)
    random_r2_scores.append(r2_random)

print('\nY-Randomization Test Results:')
print(f'Mean R² with shuffled Y: {np.mean(random_r2_scores):.4f}')
print(f'Actual RF R²: {r2:.4f}')

### **Interpretation:**
- Actual RF R² = 0.5,   
- Random R² ≈ -0.2 → Model is learning real chemical patterns, not noise.

## **LazyPredict – Compare Multiple Regression Models**

In [ ]:
print('\n' + '='*60)
print('LAZYPREDICT - COMPARING MULTIPLE REGRESSION MODELS')
print('='*60)

# Initialize LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)

# Fit all models using the filtered training data
models, predictions = reg.fit(X_train_reg_filtered, X_test_reg, y_train_reg_filtered, y_test_reg)

# Display results
print(models.head(10))  # Top models

For All Models

In [ ]:
# Display all models
print(models)

In [ ]:
models.to_csv("all_regression_models_results.csv", index=False)

In [ ]:
import pandas as pd
from google.colab import files

# Convert model index into a column
models_excel = models.reset_index()

# Rename model column
models_excel = models_excel.rename(columns={"index": "Model"})

# Save as Excel
models_excel.to_excel("LazyPredict_All_Models_With_Names.xlsx", index=False)

# Download file
files.download("LazyPredict_All_Models_With_Names.xlsx")

Optimized Machine Learning Parameters After GridSearchCV Tuning

In [ ]:
data = {
    "Machine Learning Algorithms": [
        "XGB regression Model"
    ],
    "Tuning parameters": [
        "n-estimators: 200\nRandom-state: 42\nLearning-rate: 0.1\nMax-depth: 5"
    ],
    "Result": [
        "Model performance R²: 0.59\nMean Absolute Error: 0.73\nMean Squared Error: 1.04\nRoot Mean Squared Error: 1.02\nExplained Variance Score: 0.599"
    ]
}

In [ ]:
data = {
    "Machine Learning Algorithms": [
        "Random Forest Regression"
    ],

    "Tuning parameters": [
        "n_estimators: 100\n"
        "random_state: 42\n"
        "max_depth: None\n"
        "min_samples_split: 2\n"
        "min_samples_leaf: 1"
    ],

    "Result": [
        "Adjusted R²: 0.661218\n"
        "Model performance R²: 0.706075\n"
        "RMSE: 0.971496"
    ]
}

import pandas as pd
from google.colab import files

df = pd.DataFrame(data)

df.to_excel(
    "Random_Forest_Regression_Optimized_Parameters.xlsx",
    index=False
)

files.download("Random_Forest_Regression_Optimized_Parameters.xlsx")

In [ ]:
df = pd.DataFrame(data)
file_name = "Optimized_ML_Parameters_GridSearchCV.xlsx"
df.to_excel(file_name, index=False)
# Download
files.download(file_name)

## **Visualize Top Models**

In [ ]:
top_models = models.head(10)

plt.figure(figsize=(10, 6))

# Define distinct colors for each bar
colors = ['red', 'blue', 'green', 'orange', 'purple', 'cyan', 'magenta', 'lime', 'brown', 'pink']

# Vertical bar plot with multiple colors and alpha for transparency
plt.bar(top_models.index, top_models['R-Squared'], color=colors, alpha=0.7)

plt.ylabel('R² Score', fontsize=12, fontweight='bold')
plt.xlabel('Model', fontsize=12, fontweight='bold')
plt.title('Top 10 Models from LazyPredict', fontsize=14, fontweight='bold')

# Rotate x-axis labels for readability
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the top 10 models
top_models = models.head(10)

plt.figure(figsize=(10, 6))

# Use a cohesive color palette instead of disparate colors
colors = sns.color_palette("viridis", 10)

# Create horizontal bars (barh)
# Note: .iloc[::-1] reverses the dataframe order so the highest score is at the top
plt.barh(top_models.index[::-1], top_models['R-Squared'].iloc[::-1], color=colors, alpha=0.85, edgecolor='black', linewidth=0.7)

# Labeling and styling
plt.xlabel('$R^2$ Score', fontsize=12, fontweight='bold')
plt.ylabel('Model', fontsize=12, fontweight='bold')
plt.title('Top 10 Regressors Ranked by $R^2$ Performance', fontsize=14, fontweight='bold', pad=15)
plt.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

top_models = models.head(10)
labels = top_models.index
x = np.arange(len(labels))
width = 0.35

fig, ax1 = plt.subplots(figsize=(12, 6))

# Left Y-axis: R-squared
color = '#1f77b4'
ax1.set_xlabel('Model', fontsize=12, fontweight='bold', labelpad=10)
ax1.set_ylabel('$R^2$ Score (Higher is Better)', color=color, fontsize=12, fontweight='bold')
rects1 = ax1.bar(x - width/2, top_models['R-Squared'], width, label='$R^2$', color=color, alpha=0.8)
ax1.tick_params(axis='y', labelcolor=color)

# Right Y-axis: RMSE
ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('RMSE (Lower is Better)', color=color, fontsize=12, fontweight='bold')
# Replace 'RMSE' with the exact column name in your LazyPredict output if it differs (e.g., 'RMSE')
rects2 = ax2.bar(x + width/2, top_models['RMSE'], width, label='RMSE', color=color, alpha=0.8)
ax2.tick_params(axis='y', labelcolor=color)

# Layout adjustments
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, ha='right')
plt.title('Multi-Metric Comparison of Top 10 Models', fontsize=14, fontweight='bold', pad=15)

fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get the top 10 models and reverse order for plotting top-down
top_models = models.head(10)[::-1]
names = top_models.index
r2_scores = top_models['R-Squared']

# Calculate the mean R² of the top 10 group as a benchmark
mean_r2 = np.mean(r2_scores)

plt.figure(figsize=(10, 6))

# Plot the stem lines connecting to the baseline mean
plt.hlines(y=names, xmin=mean_r2, xmax=r2_scores, color='dimgrey', alpha=0.6, linewidth=1.5)

# Plot the lollipop heads (color map scales with the score)
plt.scatter(r2_scores, names, color='#4a90e2', s=120, zorder=3, edgecolor='black', alpha=0.9)

# Add a vertical dashed line tracking the cohort average
plt.axvline(x=mean_r2, color='#e24a4a', linestyle='--', linewidth=1.5, label=f'Cohort Average R² ({mean_r2:.3f})')

# Aesthetics and formatting
plt.xlabel('$R^2$ Score', fontsize=12, fontweight='bold')
plt.ylabel('Model Architecture', fontsize=12, fontweight='bold')
plt.title('Relative Performance Ranking', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc='lower left')
plt.grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pandas.plotting import parallel_coordinates

# Select the top models and reset index to make 'Model' a column
top_models = models.head(10).copy()
top_models['Model Name'] = top_models.index

# Keep only the metrics you want to plot along with the identifier
# Adjust column names ('R-Squared', 'RMSE', 'MAE', 'Time Taken') as needed
plot_df = top_models[['Model Name', 'R-Squared', 'RMSE', 'Time Taken']]

plt.figure(figsize=(11, 6))

# Plot parallel coordinates using the model names as the grouping class
parallel_coordinates(plot_df, 'Model Name', colormap='tab10', linewidth=2.5, alpha=0.85)

# Aesthetics
plt.title('Multi-Metric Performance Trajectory Across Top 10 Regressors', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Normalized Metric Value Scale', fontsize=12, fontweight='bold')
plt.xlabel('Evaluation Criteria Axes', fontsize=12, fontweight='bold')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Model Architectures")
plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## **Compare Best Model with Random Forest**

In [ ]:
best_model_name = models.index[0]
best_r2 = models['R-Squared'].iloc[0]
print(f'Best model from LazyPredict: {best_model_name}')
print(f'Best R² score: {best_r2:.4f}')
print(f'Random Forest R²: {r2:.4f}')
print(f'Improvement: {(best_r2 - r2)*100:.2f}%')

# Check Random Forest rank
if 'RandomForestRegressor' in models.index:
    rf_rank = models.index.get_loc('RandomForestRegressor') + 1
    print(f'Random Forest ranked: {rf_rank} out of {len(models)}')

## **Random Forest Feature Importances using SHAP values**

In [ ]:
# Create an explainer and calculate SHAP values for the test set
explainer = shap.Explainer(rf_model)
shap_values = explainer(X_test_reg)

# Summary plot
shap.summary_plot(shap_values, X_test_reg, feature_names= feature_names)

In [ ]:
import matplotlib.pyplot as plt
import shap

# Generate the global absolute importance bar plot
plt.figure(figsize=(8, 5))
shap.plots.bar(shap_values, max_display=10, show=False)

# Formatting enhancements for publication
plt.title("Global Feature Importance (Mean |SHAP Value|)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Mean Absolute SHAP Value (Impact on Model Output)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import shap

# Initialize javascript visualization environment for SHAP
shap.initjs()

# Generate the stacked cohort force plot
# (Works beautifully inside Jupyter/Colab notebooks)
shap.force_plot(
    explainer.expected_value,
    shap_values.values,
    X_test_reg,
    feature_names=feature_names
)